In [2]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ================== INPUT ==================
purchase_year = 2025          # <--- Change this if you want another year
# ==========================================

df_monthly = pd.read_csv("Monthly_Rainfall_Colombo.csv")
df_monthly.columns = df_monthly.columns.str.strip().str.replace('\ufeff', '', regex=False)
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'], format='%Y-%m', errors='coerce')
df_monthly = df_monthly.dropna(subset=['Date']).reset_index(drop=True)
df_monthly['Year'] = df_monthly['Date'].dt.year.astype(int)
df_monthly['Rainfall_Colombo'] = pd.to_numeric(df_monthly['Rainfall_Colombo'], errors='coerce')
df_monthly = df_monthly.dropna(subset=['Rainfall_Colombo']).reset_index(drop=True)

df_days = pd.read_csv("Annual_Rain_Inference_Days_Colombo.csv")
df_days.columns = df_days.columns.str.strip().str.replace('\ufeff', '', regex=False)
RAINY_COL = 'No of Rainy Days_Colombo'
df_days[RAINY_COL] = pd.to_numeric(df_days[RAINY_COL], errors='coerce')

# Filter data up to previous year only
end_year = purchase_year - 1
df_hist = df_monthly[df_monthly['Year'] <= end_year].copy()

annual_rain = df_hist.groupby('Year')['Rainfall_Colombo'].sum().reset_index()

# Assign rainy days
rainy_days_dict = {}
rainy_days_dict.update({y: 101 for y in range(1995, 2011)})
rainy_days_dict.update({y: 107 for y in range(2011, 2021)})
for _, row in df_days.iterrows():
    year_str = str(row['Year']).strip()
    if '-' not in year_str:
        try:
            y = int(float(year_str))
            if y <= end_year:
                rainy_days_dict[y] = int(row[RAINY_COL])
        except:
            pass

years = annual_rain['Year'].tolist()
rainfalls = annual_rain['Rainfall_Colombo'].tolist()
rainy_days_list = [rainy_days_dict.get(y, 101) for y in years]

# Fuzzy system
rainfall_annual = ctrl.Antecedent(np.arange(0, 5001, 1), 'rainfall_annual')
rain_interf_days = ctrl.Antecedent(np.arange(0, 366, 1), 'rain_interf_days')
beta_out = ctrl.Consequent(np.arange(1.00, 1.41, 0.01), 'beta', defuzzify_method='centroid')

rainfall_annual['low']    = fuzz.trapmf(rainfall_annual.universe, [0, 0, 1000, 1750])
rainfall_annual['medium'] = fuzz.trapmf(rainfall_annual.universe, [1000, 1750, 2500, 3000])
rainfall_annual['high']   = fuzz.trapmf(rainfall_annual.universe, [2500, 3000, 5000, 5000])

rain_interf_days['low']    = fuzz.trapmf(rain_interf_days.universe, [0, 0, 30, 60])
rain_interf_days['medium'] = fuzz.trapmf(rain_interf_days.universe, [30, 60, 90, 150])
rain_interf_days['high']   = fuzz.trapmf(rain_interf_days.universe, [90, 150, 365, 365])

beta_out['no_loading']    = fuzz.trimf(beta_out.universe, [1.00, 1.00, 1.10])
beta_out['low_loading']   = fuzz.trimf(beta_out.universe, [1.00, 1.10, 1.20])
beta_out['mod_loading']   = fuzz.trimf(beta_out.universe, [1.10, 1.20, 1.30])
beta_out['high_loading']  = fuzz.trimf(beta_out.universe, [1.20, 1.30, 1.38])
beta_out['vhigh_loading'] = fuzz.trimf(beta_out.universe, [1.30, 1.38, 1.38])

rule1 = ctrl.Rule(rainfall_annual['low']    & rain_interf_days['low'],    beta_out['low_loading'])
rule2 = ctrl.Rule(rainfall_annual['low']    & rain_interf_days['medium'], beta_out['mod_loading'])
rule3 = ctrl.Rule(rainfall_annual['low']    & rain_interf_days['high'],   beta_out['mod_loading'])
rule4 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['low'],    beta_out['no_loading'])
rule5 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['medium'], beta_out['mod_loading'])
rule6 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['high'],   beta_out['high_loading'])
rule7 = ctrl.Rule(rainfall_annual['high']   & rain_interf_days['low'],    beta_out['low_loading'])
rule8 = ctrl.Rule(rainfall_annual['high']   & rain_interf_days['medium'], beta_out['high_loading'])
rule9 = ctrl.Rule(rainfall_annual['high']   & rain_interf_days['high'],   beta_out['vhigh_loading'])

beta_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
beta_sim = ctrl.ControlSystemSimulation(beta_ctrl)

# Bootstrap Monte Carlo
SIMS = 10000
SEED = 42
rng = np.random.default_rng(SEED)
beta_samples = []
for _ in range(SIMS):
    idx = rng.integers(0, len(years))
    rain = float(rainfalls[idx])
    days = float(rainy_days_list[idx])
    beta_sim.input['rainfall_annual'] = rain
    beta_sim.input['rain_interf_days'] = days
    beta_sim.compute()
    beta_samples.append(beta_sim.output['beta'])

expected_beta = np.mean(beta_samples)
print(f"Expected β for purchase year {purchase_year} (using data up to {end_year}) = {expected_beta:.4f}")

Expected β for purchase year 2025 (using data up to 2024) = 1.2531
